In [1]:
import pandas as pd
import os
from google.colab import drive


### Install necessary libraries for MySQL connection

In [2]:
!pip install pymysql sqlalchemy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 2.2 MB/s eta 0:00:00


### Mount Google Drive to access the CSV file

In [3]:
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive'


Mounted at /content/drive


### Define database configuration and CSV file path

In [4]:
DB_CONFIG = {
    "host": "mysql-olist-db-yoganinorahardian-acf4.h.aivencloud.com",
    "port": 27518,
    "user": "avnadmin",
    "password": "AVNS_veszJ1_IVpGLJ_IbS6q",
    "database": "olist_db",
}

csv_file_path = os.path.join(
    DRIVE_ROOT,
    'Purwadhika',
    'Final Project',
    'Sprint 1 - Data Foundation',
    '4. Init Projects',
    'fact_olist_order_line.csv'
)


### Read the CSV file into a Pandas DataFrame

In [5]:
try:
    df = pd.read_csv(csv_file_path)
    print(f"Successfully loaded data from {csv_file_path}")
    display(df.head())
except FileNotFoundError:
    print(f"Error: The file was not found at {csv_file_path}. Please check the path and if Google Drive is mounted correctly.")
except Exception as e:
    print(f"An error occurred while reading the CSV: {e}")


Successfully loaded data from /content/drive/MyDrive/Purwadhika/Final Project/Sprint 1 - Data Foundation/4. Init Projects/fact_olist_order_line.csv


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,item_price,freight_value,customer_id,order_status,purchase_ts,...,customer_state,product_category_name,product_category_name_english,llm_product_name_id,llm_product_name_en,seller_city,seller_state,total_order_value,delay_days,is_late
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,...,RJ,cool_stuff,cool_stuff,Mainan Edukatif Anak,Educational Toys for Kids,volta redonda,SP,72.19,-9.0,False
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,...,SP,pet_shop,pet_shop,Peralatan Dapur Modern,Modern Kitchen Equipment,sao paulo,SP,259.83,-3.0,False
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,...,MG,moveis_decoracao,furniture_decor,Dekorasi Furnitur Mewah,Luxury Furniture Decoration,borda da mata,MG,216.87,-14.0,False
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,...,SP,perfumaria,perfumery,Parfum Elegan Wanita,Elegant Women's Perfume,franca,SP,25.78,-6.0,False
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,...,SP,ferramentas_jardim,garden_tools,Peralatan Kebun Berkualitas,Quality Garden Tools,loanda,PR,218.04,-16.0,False


### Insert data into MySQL Aiven database

In [9]:
from sqlalchemy import create_engine, MetaData, Table, Column, String, BigInteger, Float, Boolean

try:
    # Create a SQLAlchemy engine
    engine = create_engine(
        f"mysql+pymysql://{DB_CONFIG['user']}:" +
        f"{DB_CONFIG['password']}@" +
        f"{DB_CONFIG['host']}:" +
        f"{DB_CONFIG['port']}/" +
        f"{DB_CONFIG['database']}"
    )

    # Table name for the fact table
    table_name = 'fact_olist_order_line'

    # Define metadata and table schema explicitly with primary key
    metadata = MetaData()
    fact_table = Table(table_name, metadata,
        Column('order_id', String(255), primary_key=True),
        Column('order_item_id', BigInteger, primary_key=True),
        Column('product_id', String(255)),
        Column('seller_id', String(255)),
        Column('shipping_limit_date', String(255)), # Store as string for datetime for simplicity
        Column('item_price', Float),
        Column('freight_value', Float),
        Column('customer_id', String(255)),
        Column('order_status', String(255)),
        Column('purchase_ts', String(255)), # Store as string for datetime for simplicity
        Column('approved_ts', String(255)),
        Column('order_delivered_carrier_date', String(255)),
        Column('delivered_customer_ts', String(255)),
        Column('estimated_delivery_ts', String(255)),
        Column('payment_type', String(255)),
        Column('review_score', Float),
        Column('review_comment_title', String(255)),
        Column('review_comment_message', String(255)),
        Column('customer_unique_id', String(255)),
        Column('customer_city', String(255)),
        Column('customer_state', String(255)),
        Column('product_category_name', String(255)),
        Column('product_category_name_english', String(255)),
        Column('llm_product_name_id', String(255)),
        Column('llm_product_name_en', String(255)),
        Column('seller_city', String(255)),
        Column('seller_state', String(255)),
        Column('total_order_value', Float),
        Column('delay_days', Float),
        Column('is_late', Boolean),
        mysql_engine='InnoDB'
    )

    # Create the table in the database if it doesn't exist
    metadata.create_all(engine)

    # Insert the DataFrame into the MySQL table
    # Now that the table schema is explicitly handled, use index=False
    df.to_sql(name=table_name, con=engine, if_exists='append', index=False)

    print(f"Data successfully inserted into '{table_name}' table in '{DB_CONFIG['database']}'.")

except Exception as e:
    print(f"An error occurred while inserting data into MySQL: {e}")

Data successfully inserted into 'fact_olist_order_line' table in 'olist_db'.
